In [1]:
import numpy as np
import pandas as pd

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.tree import DecisionTreeClassifier

In [3]:
df = pd.read_csv('titanic.csv')

In [4]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,0,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,1,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,0,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,0,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,1,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


# Let's Plan

In [5]:
df.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age             86
SibSp            0
Parch            0
Ticket           0
Fare             1
Cabin          327
Embarked         0
dtype: int64

In [6]:
df.shape

(418, 12)

In [7]:
df.drop(columns = ['PassengerId', 'Name', 'Ticket','Cabin'], inplace = True)

In [8]:
# step 1 -> train_test_split

X_train, X_test, y_train, y_test = train_test_split(df.drop(columns = ['Survived']),
                                                    df['Survived'], 
                                                    test_size = 0.2,
                                                    random_state = 2)

In [9]:
X_train.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
280,3,female,23.0,0,0,8.6625,S
284,3,female,2.0,1,1,20.2125,S
40,3,male,39.0,0,1,13.4167,C
17,3,male,21.0,0,0,7.2250,C
362,2,female,31.0,0,0,21.0000,S


In [10]:
# Imputation Transformer
trf1 = ColumnTransformer([
    ('impute_age', SimpleImputer(),[2]),
    ('impute_fare', SimpleImputer(strategy = 'most_frequent'),[5])
], remainder = 'passthrough')

In [11]:
# One hot encoding
trf2 = ColumnTransformer([
    ('ohe_sex_embarked', OneHotEncoder(sparse_output = False, handle_unknown = 'ignore'),[1,5])
], remainder = 'passthrough')

In [12]:
#scaling
trf3  = ColumnTransformer([
    ('scale', MinMaxScaler(), slice(0,8))
])

In [13]:
# Feature selection
trf4 = SelectKBest(score_func = chi2, k=2)

In [14]:
# train the model 
trf5 = DecisionTreeClassifier()

# Create Pipeline

In [15]:
pipe = Pipeline([
    ('trf1', trf1),
    ('trf2', trf2),
    ('trf3', trf3),
    ('trf4', trf4),
    ('trf5', trf5)
])

# Pipeline Vs make_pipeline

pipeline require naming of steps , make pipeline does not                                                                     
(saame applies to Column Transformer vs make_colunn_transformer)

In [16]:
# Alternate Syntax
pipe = make_pipeline(trf1, trf2, trf3, trf4, trf5)

In [17]:
# train
pipe.fit(X_train, y_train)

Pipeline(steps=[('columntransformer-1',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('impute_age', SimpleImputer(),
                                                  [2]),
                                                 ('impute_fare',
                                                  SimpleImputer(strategy='most_frequent'),
                                                  [5])])),
                ('columntransformer-2',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('ohe_sex_embarked',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  [1, 5])])),
                ('columntransformer-3',
                 ColumnTransformer(transformers=[('scale', MinMaxScaler(),
                                                  slice(0, 8, None))])),
                ('selectkbest',
                 SelectKBest(k=2,
                             score_func=<function chi2 at 0x000001B26270A700>)),
                ('decisiontreeclassifier', DecisionTreeClassifier())])

# Explore the pipeline 


In [18]:
# Code here
pipe.named_steps

{'columntransformer-1': ColumnTransformer(remainder='passthrough',
                   transformers=[('impute_age', SimpleImputer(), [2]),
                                 ('impute_fare',
                                  SimpleImputer(strategy='most_frequent'),
                                  [5])]),
 'columntransformer-2': ColumnTransformer(remainder='passthrough',
                   transformers=[('ohe_sex_embarked',
                                  OneHotEncoder(handle_unknown='ignore',
                                                sparse_output=False),
                                  [1, 5])]),
 'columntransformer-3': ColumnTransformer(transformers=[('scale', MinMaxScaler(), slice(0, 8, None))]),
 'selectkbest': SelectKBest(k=2, score_func=<function chi2 at 0x000001B26270A700>),
 'decisiontreeclassifier': DecisionTreeClassifier()}

In [19]:
# Display Pipeline

from sklearn import set_config
set_config(display = 'diagram')

In [20]:
# predict 
y_pred = pipe.predict(X_test)

In [23]:
y_pred

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

In [24]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test, y_pred)

0.6071428571428571

# Cross Validation using Pipeline

In [25]:
# cross validation using cross_val_Score
from sklearn.model_selection import cross_val_score
cross_val_score(pipe, X_train, y_train, cv=5, scoring = 'accuracy').mean()

np.float64(0.64075079149706)

# GridSearch using pipeline

In [35]:
# gridsearchv
param = {
    'decisiontreeclassifier__max_depth': [1,2,3,4,5,None]
}

In [37]:
from sklearn.model_selection import GridSearchCV
grid = GridSearchCV(pipe, param, cv=5, scoring='accuracy')
grid.fit(X_train, y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('columntransformer-1',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('impute_age',
                                                                         SimpleImputer(),
                                                                         [2]),
                                                                        ('impute_fare',
                                                                         SimpleImputer(strategy='most_frequent'),
                                                                         [5])])),
                                       ('columntransformer-2',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('ohe_sex_embarked',
                                                                         OneHotEncoder(handle_unknown='...',
                                                                                       sparse_output=False),
                                                                         [1,
                                                                          5])])),
                                       ('columntransformer-3',
                                        ColumnTransformer(transformers=[('scale',
                                                                         MinMaxScaler(),
                                                                         slice(0, 8, None))])),
                                       ('selectkbest',
                                        SelectKBest(k=2,
                                                    score_func=<function chi2 at 0x000001B26270A700>)),
                                       ('decisiontreeclassifier',
                                        DecisionTreeClassifier())]),
             param_grid={'decisiontreeclassifier__max_depth': [1, 2, 3, 4, 5,
                                                               None]},
             scoring='accuracy')

In [38]:
grid.best_score_

np.float64(0.6437358661239257)

In [39]:
grid.best_params_

{'decisiontreeclassifier__max_depth': 1}

# Exporting the pipeline 

In [45]:
# export 
import pickle
pickle.dump(pipe, open('pipe.pkl', 'wb'))